# Proof of Concept — Estimador de costos y precios de proyectos editoriales
### Sesión 4 · De las fichas al código · Stack LangChain

Este notebook es el puente entre el diseño y la implementación. En esta versión se mantiene el desarrollo lo más simple posible: usamos un CSV ficticio, SQLite y un agente mínimo que razona sobre datos estructurados.

1. Ficha 1: define el caso de uso, la política de incertidumbre y los criterios de calidad.
2. Ficha 2: prioriza la feature y marca la arquitectura por capas.
3. Ficha 3: define la fuente de datos, el tratamiento ETL y la trazabilidad.

La idea es validar el flujo antes de pensar en una versión más sofisticada.

### El mapa: cuatro capas, de abajo hacia arriba

El notebook sigue el mismo orden de la arquitectura Medallion que ya conoces: el dato fluye limpio hacia arriba y el agente solo consume lo que la capa de datos le expone.

Primero preparas el **entorno** y eliges tu modelo. Luego construyes la **capa de datos**, donde cada fuente de tu Ficha 3 se convierte en una *tool* que el agente puede llamar (RAG, SQL, APIs externas). Sobre ella montas la **capa de agente**, que es el cerebro: el *prompt* que define su rol y reglas, más la orquestación que decide qué tool usar. Y al final lo pruebas todo en un **playground** con un pequeño set de casos derivado de tu Ficha 1.

---

## 0 · Configuración del entorno

Instalamos la base común. **LangChain** aporta las abstracciones de modelo, *prompt* y agente; **langchain-community** trae las integraciones (cargadores de datos, *vector stores*, utilidades SQL); y **langsmith** te da observabilidad: registra cada paso del agente para que puedas depurarlo.

En 2026, el agente de `create_agent` corre sobre **LangGraph** por debajo, así que LangChain ya lo trae como dependencia. Más abajo dejamos comentados los *extras* que activarás según tu caso: proveedor del modelo, *vector store*, driver de base de datos y tools externas.

In [1]:
# 0.1 · Dependencias base del template
# Este PoC usa LangChain, Gemini y LangSmith como stack real.
# Si ejecutas el notebook desde cero, instala los paquetes necesarios antes de correr las celdas de agente.
%pip install -qU langchain langchain-community langchain-google-genai langsmith python-dotenv


Note: you may need to restart the kernel to use updated packages.


### 0.2 · Claves y trazabilidad

Carga las claves de forma segura (sin escribirlas en el notebook) y enciende **LangSmith**. Con la traza activa, cada ejecución de las siguientes celdas quedará registrada en tu proyecto: qué tools llamó el agente, con qué argumentos, cuánto tardó y dónde falló. Es la observabilidad de tu Ficha 3, ahora aplicada al agente.

In [2]:
# 0.2 · Configuración y rutas
import asyncio
import json
import os
import re
import sqlite3
import time
import uuid
import logging
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="Key 'additionalProperties' is not supported in schema")
logging.getLogger("langchain_google_genai._function_utils").setLevel(logging.ERROR)

import pandas as pd
from dotenv import load_dotenv

BASE_DIR = Path.cwd()
if (BASE_DIR / 'gastos_ficticios_poc.csv').exists():
    DATA_DIR = BASE_DIR
elif (BASE_DIR / 'Recursos y Contexto' / 'gastos_ficticios_poc.csv').exists():
    DATA_DIR = BASE_DIR / 'Recursos y Contexto'
elif (BASE_DIR.parent / 'Recursos y Contexto' / 'gastos_ficticios_poc.csv').exists():
    DATA_DIR = BASE_DIR.parent / 'Recursos y Contexto'
else:
    DATA_DIR = BASE_DIR / 'Recursos y Contexto'
CSV_PATH = DATA_DIR / 'gastos_ficticios_poc.csv'
DB_PATH = DATA_DIR / 'poc_gastos.sqlite'

load_dotenv(BASE_DIR / '.env')
load_dotenv(BASE_DIR.parent / '.env')

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY')
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')

if not GOOGLE_API_KEY:
    raise RuntimeError('Falta GOOGLE_API_KEY o GEMINI_API_KEY en .env')
if not LANGSMITH_API_KEY:
    raise RuntimeError('Falta LANGSMITH_API_KEY en .env')

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'PoC-ProyectoFinal-BSG')
os.environ.setdefault('LANGCHAIN_PROJECT', 'PoC-ProyectoFinal-BSG')

MARGEN_UTILIDAD = 0.35
PROJECT_ID_RE = re.compile(r'\b\d{4}-\d{4}\b')

print('Variables cargadas desde .env y trazas activadas en LangSmith.')


Variables cargadas desde .env y trazas activadas en LangSmith.


### 0.3 · Tu modelo de lenguaje

El modelo es el motor de razonamiento del agente. Elige el de tu Ficha 2. `init_chat_model` acepta un identificador con el formato `"<proveedor>:<modelo>"` y te devuelve un objeto de chat homogéneo, así puedes cambiar de proveedor tocando una sola línea.

In [3]:
# 0.3 · Modelo
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL_ID = 'gemini-2.5-flash'
llm = ChatGoogleGenerativeAI(
    model=MODEL_ID,
    temperature=0,
    thinking_budget=0,
    max_retries=5,
    metadata={'ls_model_name': MODEL_ID},
)
print(f'Modelo seleccionado: {MODEL_ID}')


Modelo seleccionado: gemini-2.5-flash


### 0.4 · Declara el stack de tu caso

Antes de codear, dejamos por escrito qué piezas usa este PoC.

- **Modelo:** agente mínimo basado en reglas sobre datos estructurados.
- **RAG (embeddings + vector store):** no aplica.
- **Base de datos (SQL):** SQLite local para pruebas.
- **Tools externas / APIs:** no aplica en esta versión.
- **Política de incertidumbre:** si falta el proyecto o la cantidad es inconsistente, el campo queda en `null` y se reporta en `campos_baja_confianza`.


## 1 · Capa de Datos: procesamiento y preparación de *tools*

Esta capa convierte tu Ficha 3 en código. La idea central: **cada fuente de datos se transforma en una capacidad que el agente puede invocar**, es decir, una *tool*. Cubrimos las tres formas más comunes —recuperación sobre texto (RAG), consulta estructurada (SQL) y herramientas externas— pero activa solo las que tu caso necesite.

Recuerda la regla de la arquitectura Medallion: el agente consume lo que esta capa **expone limpio**, nunca toca la fuente cruda directamente.

### 1.1 · RAG — recuperación sobre documentos

Convierte los documentos ya limpios de tu capa *Curated* (Ficha 3) en una *tool* de búsqueda semántica. La descripción de la tool es clave: el agente decide cuándo usarla leyéndola, así que sé específico sobre **qué contiene** y **cuándo conviene**.

In [4]:
# 1.1 · RAG
# No aplica en este PoC porque la fuente principal es estructurada y vive en SQLite.
rag_tool = None


### 1.2 · MCP local para la capa de datos

Siguiendo el patrón del laboratorio, publicamos la capa de datos como un servidor MCP local.
La idea es que LangChain descubra tools de negocio concretas, no SQL libre, y que el modelo elija la capacidad correcta según la pregunta.


In [5]:
# 1.2 · Levantar servidor MCP y descubrir tools
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_HOST = "127.0.0.1"
SERVER_PATH = Path.cwd() / "mcp_costos_editoriales_server.py"
LOG_PATH = Path.cwd() / "mcp_costos_editoriales_server.log"


def puerto_libre(host: str = "127.0.0.1") -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((host, 0))
        return sock.getsockname()[1]


def puerto_abierto(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0


MCP_PORT = puerto_libre(MCP_HOST)
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"

if not puerto_abierto(MCP_HOST, MCP_PORT):
    env = os.environ.copy()
    env["MCP_HOST"] = MCP_HOST
    env["MCP_PORT"] = str(MCP_PORT)
    env["MCP_PATH"] = "/mcp"
    log_file = open(LOG_PATH, "w")
    server_process = subprocess.Popen(
        [sys.executable, str(SERVER_PATH)],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        cwd=str(Path.cwd()),
        env=env,
    )
    for _ in range(60):
        time.sleep(0.5)
        if puerto_abierto(MCP_HOST, MCP_PORT):
            break
    else:
        raise RuntimeError(f"El servidor MCP no inició. Revisa {LOG_PATH}")
    print(f"✅ Servidor MCP activo en: {MCP_URL}")
else:
    print(f"ℹ️ Servidor MCP ya activo en: {MCP_URL}")

mcp_client = MultiServerMCPClient(
    {
        "costos_editoriales": {
            "transport": "streamable_http",
            "url": MCP_URL,
        }
    }
)

mcp_tools = await mcp_client.get_tools()
print(f"✅ Tools MCP descubiertas: {len(mcp_tools)}")


✅ Servidor MCP activo en: http://127.0.0.1:62005/mcp
✅ Tools MCP descubiertas: 13


### 1.3 · Tools MCP disponibles

Aquí se ve la capa de datos como capacidades explícitas de negocio. Las tools no exponen SQL libre; cada una responde una necesidad concreta del proyecto.


In [6]:
# 1.3 · Lista resumida de tools descubiertas
for tool in mcp_tools:
    print(f"- {tool.name}: {tool.description}")


- buscar_proyectos: Busca proyectos por idproyecto, claveproducto, nombreproducto, proveedor o tipo de gasto.

Úsala cuando el usuario no recuerde el id exacto del proyecto o quiera encontrar
proyectos relacionados con un término parcial.
- listar_proyectos_con_totales: Devuelve un listado de proyectos con cantidad de gastos y gasto total.

Úsala para ver rápidamente qué proyectos concentran más costo.
- top_proyectos_por_gasto: Devuelve los proyectos con mayor gasto total.

Úsala cuando el usuario quiera saber qué proyectos consumen más presupuesto.
- proyectos_recientes: Lista los proyectos más recientes según la fecha del proyecto.

Úsala cuando la pregunta esté orientada a actividad reciente o últimos movimientos.
- gastos_por_tipo_global: Resume el gasto de todos los proyectos agrupado por tipo de gasto.

Úsala para entender en qué categorías se concentra el presupuesto total.
- gastos_por_proveedor: Resume el gasto por proveedor.

Úsala para identificar proveedores relevantes o c

### 1.4 · Smoke test de recuperación

Probamos algunas tools MCP antes de dárselas al agente. Esto ayuda a verificar que la recuperación de datos funciona bien por sí sola.


In [7]:
# 1.4 · Pruebas directas de tools MCP
tools_por_nombre = {tool.name: tool for tool in mcp_tools}

pruebas = [
    ("listar_proyectos_con_totales", {"limite": 5}),
    ("top_proyectos_por_gasto", {"limite": 5}),
    ("resumen_proyecto_sql", {"idproyecto": "3232-0001"}),
    ("desglose_gastos_por_tipo", {"idproyecto": "3232-0001"}),
    ("gastos_por_tipo_global", {"limite": 5}),
]

for nombre, params in pruebas:
    print("─" * 80)
    print(f"🧪 {nombre}")
    print("─" * 80)
    print(await tools_por_nombre[nombre].ainvoke(params))
    print()


────────────────────────────────────────────────────────────────────────────────
🧪 listar_proyectos_con_totales
────────────────────────────────────────────────────────────────────────────────
[{'type': 'text', 'text': '[{"idproyecto":"3232-0001","claveproducto":3232,"nombreproducto":"Crece Feliz - Nueva Edicion","fechaproyecto":"2026-04-10","cantidad_gastos":6,"total_gastos":220000.0,"gasto_promedio":36666.67},{"idproyecto":"3232-0002","claveproducto":3232,"nombreproducto":"Aprender Jugando - Tercera Edicion","fechaproyecto":"2026-04-18","cantidad_gastos":5,"total_gastos":145500.0,"gasto_promedio":29100.0},{"idproyecto":"3232-0004","claveproducto":3232,"nombreproducto":"Atlas Escolar - Edicion Especial","fechaproyecto":"2026-05-20","cantidad_gastos":5,"total_gastos":120000.0,"gasto_promedio":24000.0},{"idproyecto":"3232-0003","claveproducto":3232,"nombreproducto":"Pequenos Exploradores - Reimpresion","fechaproyecto":"2026-05-05","cantidad_gastos":4,"total_gastos":90000.0,"gasto_promed

In [8]:
# 1.4 · Tools activas para el agente
# El agente no usa SQL libre: consume las tools publicadas por MCP.
tools = mcp_tools
print(f'{len(tools)} tool(s) lista(s):', [tool.name for tool in tools])


13 tool(s) lista(s): ['buscar_proyectos', 'listar_proyectos_con_totales', 'top_proyectos_por_gasto', 'proyectos_recientes', 'gastos_por_tipo_global', 'gastos_por_proveedor', 'resumen_por_estado_registro', 'resumen_proyecto_sql', 'resumen_indicadores_globales', 'detalle_gastos_proyecto', 'desglose_gastos_por_tipo', 'proyectos_con_cantidad_inconsistente', 'calcular_propuesta_costo_precio']


---

## 2 · Capa de Agente: prompt y orquestación

Ahora el cerebro. El *system prompt* traduce tu Ficha 1 a instrucciones: quién es el agente, cuál es su tarea y qué reglas respeta (sobre todo tu política de *"nunca inventar"*). La orquestación —el bucle *razonar → actuar → observar*— la arma `create_agent`, la API vigente de LangChain en 2026, que corre sobre LangGraph por debajo.

In [9]:
# 2.1 · System prompt
SYSTEM_PROMPT = """
Eres un asistente editorial especializado en costos y precios de venta.
Tu objetivo es responder usando las tools MCP disponibles sobre SQLite.

Reglas:
- Usa la herramienta más específica disponible.
- Si necesitas ubicar un proyecto, usa `buscar_proyectos`.
- Si el usuario pide un listado, usa `listar_proyectos_con_totales`, `top_proyectos_por_gasto` o `proyectos_recientes`.
- Si el usuario pide una visión global, usa `resumen_indicadores_globales`, `gastos_por_tipo_global`, `gastos_por_proveedor` o `resumen_por_estado_registro`.
- Si el usuario pide un resumen de un proyecto, usa `resumen_proyecto_sql`.
- Si quiere ver el detalle o el desglose, usa `detalle_gastos_proyecto` o `desglose_gastos_por_tipo`.
- Si la pregunta toca calidad de datos, usa `proyectos_con_cantidad_inconsistente`.
- Si el usuario pide calcular costo o precio de un proyecto, usa `calcular_propuesta_costo_precio`.
- El margen de utilidad por defecto es 35% (0.35).
- No inventes datos ni supongas valores que no estén en la base.
- Responde en lenguaje natural, breve y concreto, sin listas ni formato rígido.
- Si no tienes suficiente contexto, dilo con claridad y pide el dato faltante.
""".strip()

print(SYSTEM_PROMPT)


Eres un asistente editorial especializado en costos y precios de venta.
Tu objetivo es responder usando las tools MCP disponibles sobre SQLite.

Reglas:
- Usa la herramienta más específica disponible.
- Si necesitas ubicar un proyecto, usa `buscar_proyectos`.
- Si el usuario pide un listado, usa `listar_proyectos_con_totales`, `top_proyectos_por_gasto` o `proyectos_recientes`.
- Si el usuario pide una visión global, usa `resumen_indicadores_globales`, `gastos_por_tipo_global`, `gastos_por_proveedor` o `resumen_por_estado_registro`.
- Si el usuario pide un resumen de un proyecto, usa `resumen_proyecto_sql`.
- Si quiere ver el detalle o el desglose, usa `detalle_gastos_proyecto` o `desglose_gastos_por_tipo`.
- Si la pregunta toca calidad de datos, usa `proyectos_con_cantidad_inconsistente`.
- Si el usuario pide calcular costo o precio de un proyecto, usa `calcular_propuesta_costo_precio`.
- El margen de utilidad por defecto es 35% (0.35).
- No inventes datos ni supongas valores que no es

In [10]:
# 2.2 · Construccion del agente
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    name='poc_costos_editoriales',
)
print('Agente LangChain listo con Gemini 2.5 Flash, MCP y LangSmith.')


Agente LangChain listo con Gemini 2.5 Flash, MCP y LangSmith.


### ¿Cuándo subir de `create_agent` a LangGraph?

`create_agent` te da el bucle *razonar → actuar → observar* listo para usar y alcanza para la mayoría de los PoC. El momento de bajar a un `StateGraph` explícito de LangGraph llega cuando tu caso necesita ramas condicionales, reintentos basados en resultados intermedios, una pausa para aprobación humana antes de una acción sensible, o memoria persistente entre sesiones. No empieces ahí: súbete de nivel cuando el PoC te lo pida.

---

## 3 · Test y Playground

Probamos el agente de tres formas: una invocación simple, un pequeño set de casos derivado de tu Ficha 1 (la meta no es *"que suene bien"*, sino verificar el comportamiento esperado), y un playground interactivo para explorar.

In [11]:
# 3.1 · Invocacion simple
def extraer_texto_respuesta(salida):
    if hasattr(salida, 'content'):
        return salida.content
    if isinstance(salida, dict):
        mensaje_final = salida['messages'][-1]
        if hasattr(mensaje_final, 'content'):
            return mensaje_final.content
        return mensaje_final['content']
    return str(salida)


async def preguntar(texto: str, max_intentos: int = 5) -> str:
    ultimo_error = None
    for intento in range(max_intentos):
        try:
            salida = await agent.ainvoke({'messages': [{'role': 'user', 'content': texto}]})
            return extraer_texto_respuesta(salida)
        except Exception as e:
            mensaje = str(e)
            if '503' in mensaje or 'UNAVAILABLE' in mensaje or 'contents are required' in mensaje:
                ultimo_error = e
                espera = 2 ** intento
                print(f'  ⏳ Intento {intento + 1} falló, reintentando en {espera}s... ({type(e).__name__})')
                await asyncio.sleep(espera)
            else:
                raise
    raise RuntimeError(f'Se agotaron los reintentos. Último error: {ultimo_error}')

print(await preguntar('Calcula el costo y precio del proyecto 3232-0001'))


El costo propuesto para el proyecto 3232-0001 es de 220000.0 y el precio sugerido es de 297000.0, aplicando un margen de utilidad del 35%.


In [12]:
# 3.2 · Mini set de prueba
casos = [
    'Resume el proyecto 3232-0001',
    'Calcula el costo y precio del proyecto 3232-0001',
    '¿Qué proyectos tienen mayor gasto?',
    'Muestra los proyectos recientes',
    '¿Hay proyectos con cantidades inconsistentes?',
    '¿Cuál es el resumen global de la base?',
]
for i, c in enumerate(casos, 1):
    print(f'\n=== Caso {i} ===\n{c}\n---')
    print(await preguntar(c))



=== Caso 1 ===
Resume el proyecto 3232-0001
---
El proyecto 3232-0001, "Crece Feliz - Nueva Edicion", tiene un gasto total de 220,000.00, distribuido en 6 gastos distintos. La fecha del proyecto es 10 de abril de 2026, con gastos registrados entre el 26 de marzo de 2026 y el 4 de abril de 2026. El gasto promedio por ítem es de 36,666.67.

=== Caso 2 ===
Calcula el costo y precio del proyecto 3232-0001
---
El costo propuesto para el proyecto 3232-0001 (Crece Feliz - Nueva Edicion) es de 220,000.0 y el precio sugerido es de 297,000.0, aplicando un margen de utilidad del 35%.

=== Caso 3 ===
¿Qué proyectos tienen mayor gasto?
---
Los proyectos con mayor gasto son "Crece Feliz - Nueva Edicion" (220000.0), "Aprender Jugando - Tercera Edicion" (145500.0) y "Atlas Escolar - Edicion Especial" (120000.0).

=== Caso 4 ===
Muestra los proyectos recientes
---
Los proyectos recientes son: Atlas Escolar - Edicion Especial (ID: 3232-0004, Fecha: 2026-05-20, Gasto Total: 120000.0), Pequeños Explorado

In [14]:
# 3.3 · Playground interactivo
import asyncio

async def playground():
    print("Playground interactivo. Escribe 'salir', 'exit' o 'quit' para terminar.")
    while True:
        q = await asyncio.to_thread(input, '\nTu > ')
        if q is None or q.strip().lower() in {'salir', 'exit', 'quit'}:
            break
        texto = q.strip()
        if not texto:
            continue
        try:
            print('Agente >', await preguntar(texto))
        except ValueError as e:
            if 'contents are required' in str(e):
                print('  ⚠️  La sesión fue interrumpida. Escribe de nuevo.')
                continue
            raise

await playground()


Playground interactivo. Escribe 'salir', 'exit' o 'quit' para terminar.
Agente > Los proyectos más recientes son: "Atlas Escolar - Edicion Especial" (ID: 3232-0004) con un gasto de 120000.0, "Pequenos Exploradores - Reimpresion" (ID: 3232-0003) con 90000.0, "Aprender Jugando - Tercera Edicion" (ID: 3232-0002) con 145500.0, y "Crece Feliz - Nueva Edicion" (ID: 3232-0001) con 220000.0.
Agente > El costo propuesto para el proyecto 3232-0002 es de 145500.0 y el precio sugerido con un margen de utilidad del 35% es de 196425.0.
Agente > No se encontraron detalles de gastos para el proyecto 3232-000905.


### 3.4 · Observabilidad

Si activaste LangSmith en el paso 0.2, todas las ejecuciones de arriba quedaron registradas en tu proyecto en [smith.langchain.com](https://smith.langchain.com). Ahí puedes ver el árbol de cada corrida: qué tools llamó el agente, con qué argumentos, la latencia por paso y dónde falló. Úsalo para depurar el comportamiento, no solo el resultado final.

---